# ML-08 — Capstone Modeling Lane

This notebook tests a learned ranking model against my Week-4 hand-written baseline on the same anonymized teaching dataset.

**Goal:** rank pages for content-review decision support. This is not a claim about Google's ranking algorithm.


## 1. Method choice and why

I use a **Random Forest classifier** because this lane is a scoring/ranking problem with mixed numeric and categorical signals. It can capture non-linear interactions among visibility, position, freshness, CTR, and content characteristics.

I exclude `trend_direction` and `trend_pct` because they are directly related to the target label. This avoids target leakage.


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_PATH = Path(os.environ.get("FLYRANK_DATA_PATH", "data/raw/content_refresh_anonymized.csv"))
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded local dataset: {DATA_PATH}")
else:
    DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(DATA_URL)
    print("Loaded the public anonymized FlyRank teaching dataset.")

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates("content_id").reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows used: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique():,}")
print(f"Decline base rate: {df['is_declining_label'].mean():.2%}")


Loaded the public anonymized FlyRank teaching dataset.
Rows used: 30,000
Unique clients: 32
Decline base rate: 54.21%


## 2. Split design

I use an **80/20 client-group holdout**. All pages from a client stay in either train or test, so the model is evaluated on clients it did not see during training.

The Week-4 baseline is recomputed on the **same test rows** and evaluated with the same metrics.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, y=df["is_declining_label"], groups=df["client_id"]))

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

assert set(train["client_id"]).isdisjoint(set(test["client_id"]))

print(f"Train rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"Train clients: {train['client_id'].nunique():,}")
print(f"Test clients: {test['client_id'].nunique():,}")
print(f"Train decline rate: {train['is_declining_label'].mean():.2%}")
print(f"Test decline rate: {test['is_declining_label'].mean():.2%}")


Train rows: 23,837
Test rows: 6,163
Train clients: 25
Test clients: 7
Train decline rate: 55.01%
Test decline rate: 51.10%


## 3. Train + compare vs my baseline

The Week-4 baseline uses the same transparent rule: 40% visibility, 30% freshness risk, 25% position opportunity, and 5% depth gap.

The learned model is trained only on the training clients. Both methods are compared on the held-out test clients using ROC AUC, Average Precision, Precision@20, and Precision@50.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

def normalize(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi != lo else s * 0

def baseline_score(frame):
    x = frame.copy()
    x["visibility_score"] = percentile_rank(np.log1p(x["impressions_90d"]))
    x["freshness_risk_score"] = percentile_rank(x["days_since_last_update"])
    pos = x["avg_position"].clip(lower=1, upper=50)
    x["position_opportunity_score"] = (1 - normalize(pos)) * x["visibility_score"] * (x["avg_position"] > 0).astype(int)
    x["depth_gap_score"] = (1 - percentile_rank(x["word_count"])) * x["visibility_score"]
    return (0.40*x["visibility_score"] + 0.30*x["freshness_risk_score"] +
            0.25*x["position_opportunity_score"] + 0.05*x["depth_gap_score"]).clip(0,1)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

baseline_test_score = baseline_score(test)

numeric_features = [
    "search_volume","cpc","word_count","char_count","impressions_90d",
    "clicks_90d","sessions_90d","ai_sessions_90d",
    "days_with_impressions_90d","days_with_sessions_90d",
    "content_age_days","days_since_last_update","ctr","avg_position",
    "engagement_rate","scroll_rate","ai_traffic_pct"
]
categorical_features = [
    "competition","content_type","main_intent","competition_level",
    "age_tier","freshness_tier","word_count_tier","impression_tier","position_tier"
]

available_numeric = [c for c in numeric_features if c in df.columns]
available_categorical = [c for c in categorical_features if c in df.columns]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), available_numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), available_categorical)
])

model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1
    ))
])

X_train = train[available_numeric + available_categorical]
y_train = train["is_declining_label"]
X_test = test[available_numeric + available_categorical]
y_test = test["is_declining_label"]

model.fit(X_train, y_train)
model_test_score = model.predict_proba(X_test)[:,1]

comparison = pd.DataFrame({
    "method":["Week-4 baseline","Random Forest"],
    "ROC AUC":[roc_auc_score(y_test, baseline_test_score), roc_auc_score(y_test, model_test_score)],
    "Average Precision":[average_precision_score(y_test, baseline_test_score), average_precision_score(y_test, model_test_score)],
    "Precision@20":[precision_at_k(y_test, baseline_test_score, 20), precision_at_k(y_test, model_test_score, 20)],
    "Precision@50":[precision_at_k(y_test, baseline_test_score, 50), precision_at_k(y_test, model_test_score, 50)]
}).round(3)

display(comparison)


,method,ROC AUC,Average Precision,Precision@20,Precision@50
0,Week-4 baseline,0.502,0.484,0.35,0.32
1,Random Forest,0.600,0.587,0.55,0.64


### Reading the comparison

Use the generated table as the measured result. The key check is whether the Random Forest concentrates more observed declining pages near the top of the **same held-out test set** than the Week-4 rule.

These measurements are specific to this anonymized teaching dataset and this client-holdout split. They are not a guarantee of future performance.


## 4. Errors and interpretation

I check false positives and false negatives at a practical threshold, then inspect permutation importance on a bounded held-out sample.

A false positive is a page the model scores as declining but whose observed label is not declining. A false negative is a declining page that receives a lower model score. These are evaluation errors, not causal conclusions.


In [4]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix

threshold = 0.50
pred = (model_test_score >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

error_summary = pd.DataFrame({
    "error_type":["True positives","False positives","False negatives","True negatives"],
    "count":[tp,fp,fn,tn]
})
display(error_summary)

sample_n = min(1000, len(X_test))
sample = X_test.sample(sample_n, random_state=42)
sample_y = y_test.loc[sample.index]

perm = permutation_importance(
    model, sample, sample_y,
    scoring="average_precision", n_repeats=3,
    random_state=42, n_jobs=-1
)
importance = pd.Series(perm.importances_mean, index=sample.columns).sort_values(ascending=False)
display(importance.head(10).rename("mean_permutation_importance").to_frame())

print(f"At threshold {threshold:.2f}: {fp:,} false positives and {fn:,} false negatives.")


,error_type,count
0,True positives,1929
1,False positives,1363
2,False negatives,1220
3,True negatives,1651


,mean_permutation_importance
impressions_90d,0.039604
avg_position,0.014235
position_tier,0.009361
impression_tier,0.008573
search_volume,0.005838
content_age_days,0.005829
clicks_90d,0.005169
sessions_90d,0.004828
engagement_rate,0.004575
ctr,0.001786


At threshold 0.50: 1,363 false positives and 1,220 false negatives.


### Interpretation

Permutation importance describes **feature reliance**, not causation. A feature that matters to the model's ranking does not mean changing that feature will itself cause improvement.

The output is therefore a measured ranking signal for human review, not an automatic content decision.


In [5]:
final_table = comparison.copy()
final_table["test_clients"] = test["client_id"].nunique()
final_table["test_rows"] = len(test)
display(final_table)

forbidden = {"trend_direction","trend_pct","is_declining_label"}
used_features = set(available_numeric + available_categorical)
assert not (used_features & forbidden)
assert set(train["client_id"]).isdisjoint(set(test["client_id"]))

print("Self-checks passed: no target columns in features and no client overlap between train/test.")


,method,ROC AUC,Average Precision,Precision@20,Precision@50,test_clients,test_rows
0,Week-4 baseline,0.502,0.484,0.35,0.32,7,6163
1,Random Forest,0.600,0.587,0.55,0.64,7,6163


Self-checks passed: no target columns in features and no client overlap between train/test.


## Research reading note

I also reviewed the FlyRank research paper **The State of AI-Driven SEO in Numbers (March 2026)** as context for this modeling work.

One reported finding is that growing content was observed to be **37.6% longer** and **20% younger** than declining content in the paper's portfolio analysis. I treat this as observational evidence, not as a causal rule. The paper's portfolio-level findings should not be substituted for the held-out evaluation in this notebook.

This matches the `training-honest-models` guidance: keep the model comparison on the same split and metrics, read errors before trusting scores, use fixed seeds, and treat feature importance as model reliance rather than causation.


### Skill checks used

- `training-honest-models`: method fits the question, baseline and model use the same held-out split and metrics, errors are inspected, random seed is fixed, and top features are interpreted cautiously.
- `flyrank/flyrank-data`: uses the 30,000-row anonymized starter CSV, keeps the page/client grain explicit, excludes target-derived fields from features, and avoids client names, URLs, private queries, and product decision outputs.


## Self-check

- [x] Method choice explained.
- [x] Client-grouped train/test split.
- [x] Week-4 baseline evaluated on the same test set.
- [x] Same ranking metrics used for baseline and model.
- [x] Target/leakage columns excluded from model features.
- [x] Errors and feature reliance inspected.
- [x] Claims use careful words: observed, measured, directional, decision-support.
- [x] No client names, URLs, or private queries.
- [x] Run top to bottom once and commit the executed notebook here.
